# 16 — Spain comparison

Analyse Portugal–Spain pre-tax price spreads and annual physical-balance differences. Spain is a comparison series, not automatically a valid causal control.


In [ ]:
from pathlib import Path
import sys

ROOT = Path.cwd()
if not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from portugal_refining_resilience.config import get_paths, load_analysis_config
from portugal_refining_resilience.io import persist_dataframe, write_json

PATHS = get_paths(ROOT)
CONFIG = load_analysis_config(ROOT)
START_YEAR, END_YEAR = int(CONFIG["start_year"]), int(CONFIG["end_year"])
pd.set_option("display.max_columns", 100)

# Price spreads are descriptive; physical comparisons come from Eurostat when available.


In [ ]:
price_path = PATHS.interim / "weekly_oil_prices_tidy.csv"
if not price_path.exists():
    raise FileNotFoundError("Run the weekly price extraction before the Spain comparison.")
prices = pd.read_csv(price_path, parse_dates=["date"])
rows = []
for product in ["diesel", "gasoline"]:
    sub = prices.loc[prices["product"] == product]
    wide = sub.pivot(index="date", columns="country", values="price_without_tax_eur_per_1000l").dropna(subset=["PT", "ES"]).reset_index()
    wide["spread"] = wide["PT"] - wide["ES"]
    wide["post"] = (wide["date"] >= pd.Timestamp("2021-05-01")).astype(int)
    pre_mean = wide.loc[wide["post"] == 0, "spread"].mean()
    post_mean = wide.loc[wide["post"] == 1, "spread"].mean()
    rows.append({"product": product, "pre_spread_mean": pre_mean, "post_spread_mean": post_mean, "difference": post_mean - pre_mean, "unit": "EUR/1000L", "interpretation": "descriptive PT-ES spread change"})
    fig, ax = plt.subplots(figsize=(11, 4))
    ax.plot(wide["date"], wide["spread"])
    ax.axvline(pd.Timestamp("2021-05-01"), linestyle="--", linewidth=1)
    ax.axhline(0, linewidth=1)
    ax.set(title=f"Portugal minus Spain pre-tax {product} price", ylabel="EUR/1000L", xlabel="Date")
    fig.tight_layout()
    fig.savefig(PATHS.figures / f"pt_es_{product}_pretax_price_spread.png", dpi=180)
    plt.show()
spread_summary = pd.DataFrame(rows)
persist_dataframe(spread_summary, PATHS.metrics / "pt_es_price_spread_summary.csv")
display(spread_summary)


In [ ]:
balance_path = PATHS.processed / "eurostat_physical_balance_panel.csv"
if balance_path.exists():
    balance = pd.read_csv(balance_path)
    # Eurostat publishes past the study window, and this comparison used to take
    # whatever it shipped. That put a year in the evidence bundle that the paper
    # excludes, so a reader checking a claim against the bundle saw a series the
    # analysis never used.
    balance = balance.loc[balance["year"].between(START_YEAR, END_YEAR)]
    ratio_columns = [
        "gross_import_dependence",
        "net_import_to_demand_ratio",
        "refinery_output_to_demand_ratio",
    ]
    rows = []
    for product, sub in balance.groupby("product"):
        for ratio in ratio_columns:
            if ratio not in sub.columns:
                continue
            wide = sub.pivot(index="year", columns="country", values=ratio).dropna(subset=["PT", "ES"])
            for year, row in wide.iterrows():
                rows.append({"year": int(year), "product": product, "metric": ratio, "PT": row["PT"], "ES": row["ES"], "PT_minus_ES": row["PT"] - row["ES"]})
    physical_comparison = pd.DataFrame(rows)
    persist_dataframe(
        physical_comparison,
        PATHS.metrics / "pt_es_physical_balance_comparison.csv",
        key_columns=["year", "product", "metric"],
    )
    display(physical_comparison.tail())
else:
    print("Missing data/processed/eurostat_physical_balance_panel.csv; PT-ES physical-balance comparison remains incomplete.")


### Control validity checks for the report

Before using causal language, inspect pre-2021 trends, common tax changes, product-definition consistency, COVID-period distortions and the degree to which Spain experienced similar refinery/logistics shocks. If these diagnostics are weak, retain descriptive language.


In [ ]:
# Table 12 shows 2018-2024 because that is the span being compared. Spain is not flat
# outside it, and showing only the flat-looking years is a selection a reader is entitled
# to see for themselves.
_comparison = pd.read_csv(PATHS.metrics / "pt_es_physical_balance_comparison.csv")
_ratio = _comparison.loc[
    (_comparison["metric"] == "refinery_output_to_demand_ratio")
    & (_comparison["product"] == "diesel")
].sort_values("year")
fig, ax = plt.subplots(figsize=(11, 4))
ax.plot(_ratio["year"], _ratio["PT"], label="Portugal")
ax.plot(_ratio["year"], _ratio["ES"], label="Spain")
ax.axhline(1.0, linewidth=1, linestyle=":")
ax.axvline(2013, linestyle="--", linewidth=1)
ax.axvline(2021, linestyle="--", linewidth=1)
ax.axvspan(2018, 2024, alpha=0.08)
ax.set(
    title="Diesel refinery output to demand, Portugal and Spain, full window",
    ylabel="ratio",
    xlabel="year",
)
ax.legend()
fig.tight_layout()
fig.savefig(PATHS.figures / "pt_es_diesel_output_ratio_full_window.png", dpi=180)
plt.close(fig)
print("full-window Portugal-Spain coverage figure written")
